In [1]:
import re
import math
import random
from collections import Counter, defaultdict

In [2]:
def to_sentences(text):
    return [
        s.strip()
        for s in re.split(r"[.!?]+", text)
        if s.strip()
    ]

def to_tokens(sentence):
    return re.findall(r"[a-z']+", sentence.lower())

In [3]:
def build(sents, n):
    """Map each (n-1)-word context to a Counter of following words."""
    foll = defaultdict(Counter)

    for s in sents:
        toks = ["<s>"] * (n - 1) + to_tokens(s) + ["</s>"]

        for i in range(len(toks) - n + 1):
            context = tuple(toks[i:i + n - 1])
            next_word = toks[i + n - 1]
            foll[context][next_word] += 1

    return foll

In [4]:
class NgramLM:
    """A smoothed n-gram language model."""

    def __init__(self, n=2, k=0.5):
        self.n = n
        self.k = k

    def fit(self, text):
        self.sents = to_sentences(text)
        self.foll = build(self.sents, self.n)

        self.V = (
            set(t for s in self.sents for t in to_tokens(s))
            | {"</s>"}
        )

        return self

    def _prob(self, ctx, w):
        c = self.foll.get(ctx, Counter())
        total = sum(c.values())

        return (
            (c[w] + self.k) /
            (total + self.k * len(self.V))
        )

    def suggest_next(self, text, topk=3):
        ctx = tuple(
            (["<s>"] * (self.n - 1) + to_tokens(text))
            [-(self.n - 1):]
        )

        c = self.foll.get(ctx, Counter())

        words = [
            w for w, _ in c.most_common()
            if w != "</s>"
        ]

        return words[:topk] or ["(no idea)"]

    def generate(self, seed=0, max_len=25):
        rng = random.Random(seed)

        ctx = ("<s>",) * (self.n - 1)
        out = []

        for _ in range(max_len):
            c = self.foll.get(ctx)

            if not c:
                break

            words, weights = zip(*c.items())
            nxt = rng.choices(words, weights=weights)[0]

            if nxt == "</s>":
                break

            out.append(nxt)
            ctx = (ctx + (nxt,))[-(self.n - 1):]

        return " ".join(out)

    def perplexity(self, text):
        log_sum = 0.0
        N = 0

        for s in to_sentences(text):
            toks = (
                ["<s>"] * (self.n - 1)
                + to_tokens(s)
                + ["</s>"]
            )

            for i in range(len(toks) - self.n + 1):
                ctx = tuple(toks[i:i + self.n - 1])
                w = toks[i + self.n - 1]

                log_sum += math.log(self._prob(ctx, w))
                N += 1

        return math.exp(-log_sum / N)

In [5]:
with open("corpus.txt", "r") as file:
    text = file.read()

lm = NgramLM(n=2, k=0.5).fit(text)

print("Model trained!")
print("N-gram size:", lm.n)
print("Vocabulary size:", len(lm.V))
print("Number of contexts:", len(lm.foll))

Model trained!
N-gram size: 2
Vocabulary size: 170
Number of contexts: 170


In [6]:
for phrase in [
    "i drink",
    "the morning",
    "on the",
    "she reads a"
]:
    print(f"{phrase!r:15s}-> {lm.suggest_next(phrase)}")

'i drink'      -> ['coffee', 'tea']
'the morning'  -> ['train', 'coffee', 'before']
'on the'       -> ['morning', 'weekend', 'park']
'she reads a'  -> ['good', 'book', 'film']


In [7]:
print("Generated:", lm.generate(seed=3))

Generated: the quiet and coffee


In [8]:
print("Corpus perplexity:", lm.perplexity(text))

Corpus perplexity: 39.72969224997387


In [9]:
rng = random.Random(0)
sents = to_sentences(text)
rng.shuffle(sents)

cut = int(0.8 * len(sents))
train_text = ". ".join(sents[:cut])
test_text = ". ".join(sents[cut:])

lm_test = NgramLM(n=2, k=0.5).fit(train_text)

print("Training perplexity:", lm_test.perplexity(train_text))
print("Held-out perplexity:", lm_test.perplexity(test_text))

Training perplexity: 35.30724462198419
Held-out perplexity: 83.1219971256845


In [10]:
bigram = NgramLM(n=2, k=0.5).fit(text)
trigram = NgramLM(n=3, k=0.5).fit(text)

print("BIGRAM")
print("Suggestion:", bigram.suggest_next("the morning"))
print("Generated:", bigram.generate(seed=3))

print("\nTRIGRAM")
print("Suggestion:", trigram.suggest_next("the morning"))
print("Generated:", trigram.generate(seed=3))

BIGRAM
Suggestion: ['train', 'coffee', 'before']
Generated: the quiet and coffee

TRIGRAM
Suggestion: ['train', 'helps', 'coffee']
Generated: the garden is full of flowers in the park is quiet and full of old books


In [11]:
print("Bigram held-out:", bigram.perplexity(test_text))
print("Trigram held-out:", trigram.perplexity(test_text))

Bigram held-out: 40.937424397916764
Trigram held-out: 48.865474613344084


In [12]:
bigram = NgramLM(n=2, k=0.5).fit(train_text)
trigram = NgramLM(n=3, k=0.5).fit(train_text)

print("Bigram:", bigram.perplexity(test_text))
print("Trigram:", trigram.perplexity(test_text))

Bigram: 83.1219971256845
Trigram: 108.85444198342513
